In [ ]:
import GeoPandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

### Explanation of location geojson

GeoJSON is composed of "features" which each represent a power line  

The geometry is linestring because its line consisting of ordered coordinate points  

Properties shows what type the object is (major or minor power line) and voltage (the others can be ignored)

```json
  {
    "type": "Feature",
    "geometry": {
      "type": "LineString",
      "coordinates": [
        [
          -92.2619846,
          39.0210987
        ],
        [
          -92.2613601,
          39.0211232
        ],
        [
          -92.2614052,
          39.0205361
        ]
      ]
    },
    "properties": {
      "cables": "3",
      "frequency": "60",
      "power": "line",
      "voltage": "69000",
      "wires": "single"
    }
  },
  ```

### Importing data from overpass

line: major overhead transmission lines  
minor_line: smaller overhead distribution lines

In [5]:
import requests, json

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
query = """
[out:json][timeout:60];

area["name"="Columbia"]["boundary"="administrative"]->.sc;

(
  way["power"="line"](area.sc);
  way["power"="minor_line"](area.sc);
);

out geom;
"""
resp = requests.post(OVERPASS_URL, data=query, headers={"Content-Type":"text/plain"})
resp.raise_for_status()
data = resp.json()         


Converting powerlines into GeoJSON

In [6]:
features = []

for el in data.get("elements", []):
    if el["type"] == "way" and "geometry" in el:
        coords = [[pt["lon"], pt["lat"]] for pt in el["geometry"]]

        feature = {
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": coords
            },
            "properties": el.get("tags", {})
        }

        features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "features": features
}

with open("south_carolina_power_lines.geojson", "w") as f:
    json.dump(geojson, f, indent=2)

print("Saved GeoJSON with", len(features), "features")

Saved GeoJSON with 386 features


Read back in the tree json

In [ ]:
lines_json = "GridstormHacks-/proximity/south_carolina_power_lines.geojson"

# read in the tree canopy data
lines = gpd.read_file(lines_json)

Get the tree data

In [ ]:
hdf_path = "trees.h5" 

df_trees = load_trees_from_hdf(hdf_path, table_key=table_key, lon_col="lon", lat_col="lat")

# create a GeoDataFrame from lon/lat
trees = gpd.GeoDataFrame(
    df_trees,
    geometry=gpd.points_from_xy(df_trees["lon"], df_trees["lat"]),
    crs="EPSG:4326"
)

Reprojecting both lines and trees onto CRS (Coordinate Reference System)

In [ ]:
metric_crs = "EPSG:3857"
trees_m = trees.to_crs(metric_crs)
lines_m = lines.to_crs(metric_crs)

Creating a power line buffer (proximity zone)

In [ ]:
# buffer lines by 10 meters 
lines_buffered = lines_m.copy()
lines_buffered["geometry"] = lines_buffered.geometry.buffer(10)

Find trees within proximity zone

In [ ]:
# spatial join trees that intersect any buffered line
tree_candidates = gpd.sjoin(trees_m, lines_buffered[["geometry"]], how="inner", predicate="intersects")

Print the intersecting trees

In [ ]:
print(f"Found {len(tree_candidates)} trees within 10 m of power lines")

Save trees to GeoJSON

In [ ]:
tree_candidates.to_file("trees_near_powerlines.geojson", driver="GeoJSON")